# Postprocessing detect

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/postprocessing/Document_Postprocessing_Detect.ipynb)


## Jump to an API

- [postprocessing.select](#postprocessingselect)
- [postprocessing.map_boxes](#postprocessingmap_boxes)
- [postprocessing.relabel](#postprocessingrelabel)
- [postprocessing.drop](#postprocessingdrop)
- [postprocessing.filter_score](#postprocessingfilter_score)
- [postprocessing.clip_boxes](#postprocessingclip_boxes)
- [postprocessing.remove_invalid_boxes](#postprocessingremove_invalid_boxes)
- [postprocessing.box_iou](#postprocessingbox_iou)
- [postprocessing.suppress](#postprocessingsuppress)
- [postprocessing.top_boxes](#postprocessingtop_boxes)
- [postprocessing.threshold](#postprocessingthreshold)
- [postprocessing.clip](#postprocessingclip)
- [postprocessing.remove_invalid](#postprocessingremove_invalid)
- [postprocessing.nms](#postprocessingnms)
- [postprocessing.top](#postprocessingtop)

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@8e42b64

In [ ]:
import tempfile
from PIL import Image
from klygo import postprocessing as post, files
from klygo.outputs.detect import Box, Detection, Detections
tutorial_workspace = tempfile.TemporaryDirectory()
root = files.path(tutorial_workspace.name)
image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])


### postprocessing.select

`postprocessing.select(result: 'Result', *, ids: 'Optional[Iterable[int]]' = None, uids: 'Optional[Iterable[str]]' = None, labels: 'Optional[Iterable[str]]' = None, where: 'Optional[Predicate]' = None) -> 'Result'`

Keep boxes matching all supplied selectors.

- Parameters: result is a Detection or Detections; ids, uids, labels, and where narrow the selection.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.select(frame, labels=["cat"]).labels)

**Practical variation**

In [ ]:
print(post.select(frame, ids=[1]).labels)

**Expected error and recovery**

In [ ]:
try:
    post.select(object())
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.map_boxes

`postprocessing.map_boxes(result: 'Result', function: 'Callable[[Box], Optional[Box]]') -> 'Result'`

Transform boxes and optionally discard them by returning None.

- Parameters: function receives a Box and returns Box or None.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.map_boxes(frame, lambda box: box.with_label("animal")).labels)

**Practical variation**

In [ ]:
print(post.map_boxes(frame, lambda box: None if box.score < 0.5 else box).labels)

**Expected error and recovery**

In [ ]:
try:
    post.map_boxes(frame, lambda box: 42)
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.relabel

`postprocessing.relabel(result: 'Result', label: 'Union[str, Mapping[int, str], Callable[[Box], str]]', *, ids: 'Optional[Iterable[int]]' = None, uids: 'Optional[Iterable[str]]' = None, labels: 'Optional[Iterable[str]]' = None, where: 'Optional[Predicate]' = None) -> 'Result'`

Change labels for selected boxes while retaining the rest.

- Parameters: label is a string, ID mapping, or callable; selectors restrict affected boxes.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.relabel(frame, "feline", labels=["cat"]).labels)

**Practical variation**

In [ ]:
print(post.relabel(frame, {0: "feline"}).labels)

**Expected error and recovery**

In [ ]:
try:
    post.relabel(object(), 'new')
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.drop

`postprocessing.drop(result: 'Result', *, ids: 'Optional[Iterable[int]]' = None, uids: 'Optional[Iterable[str]]' = None, labels: 'Optional[Iterable[str]]' = None, where: 'Optional[Predicate]' = None) -> 'Result'`

Remove boxes matching selectors.

- Parameters: ids, uids, labels, and where identify boxes to remove.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.drop(frame, labels=["dog"]).labels)

**Practical variation**

In [ ]:
print(post.drop(frame, where=lambda box: box.score < 0.5).labels)

**Expected error and recovery**

In [ ]:
try:
    post.drop(object())
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.filter_score

`postprocessing.filter_score(result: 'Result', minimum: 'float') -> 'Result'`

Keep boxes whose confidence reaches the minimum.

- Parameters: minimum is the inclusive score threshold.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.filter_score(frame, 0.5).labels)

**Practical variation**

In [ ]:
print(post.filter_score(frame, 0.8).labels)

**Expected error and recovery**

In [ ]:
try:
    post.filter_score(frame, 'bad')
except ValueError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.clip_boxes

`postprocessing.clip_boxes(result: 'Result', image_size: 'Optional[Tuple[int, int]]' = None) -> 'Result'`

Clip box coordinates to source-image or explicit bounds.

- Parameters: image_size optionally overrides source dimensions.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.clip_boxes(frame)[0].box)

**Practical variation**

In [ ]:
print(post.clip_boxes(frame, image_size=(80, 60))[0].box)

**Expected error and recovery**

In [ ]:
try:
    post.clip_boxes(object())
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.remove_invalid_boxes

`postprocessing.remove_invalid_boxes(result: 'Result', min_area: 'float' = 0) -> 'Result'`

Remove boxes with invalid coordinates or too little area.

- Parameters: min_area is the smallest retained box area.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(len(post.remove_invalid_boxes(frame, min_area=100)))

**Practical variation**

In [ ]:
print(len(post.remove_invalid_boxes(frame, min_area=2000)))

**Expected error and recovery**

In [ ]:
try:
    post.remove_invalid_boxes(frame, 'bad')
except ValueError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.box_iou

`postprocessing.box_iou(first: 'Union[Box, Sequence[float]]', second: 'Union[Box, Sequence[float]]') -> 'float'`

Compute intersection over union of two XYXY boxes.

- Parameters: first and second are Box objects or four-number XYXY sequences.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(round(post.box_iou(frame[0], frame[1]), 3))

**Practical variation**

In [ ]:
print(post.box_iou([0, 0, 10, 10], [20, 20, 30, 30]))

**Expected error and recovery**

In [ ]:
try:
    post.box_iou([1, 2, 3], [1, 2, 3, 4])
except ValueError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.suppress

`postprocessing.suppress(result: 'Result', iou: 'float' = 0.5, class_agnostic: 'bool' = False) -> 'Result'`

Apply nonmaximum suppression to overlapping boxes.

- Parameters: iou is overlap threshold; class_agnostic merges class groups when true.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.suppress(frame, iou=0.5).labels)

**Practical variation**

In [ ]:
print(post.suppress(frame, iou=0.5, class_agnostic=True).labels)

**Expected error and recovery**

In [ ]:
try:
    post.suppress(frame, iou=1.2)
except ValueError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.top_boxes

`postprocessing.top_boxes(result: 'Result', count: 'int') -> 'Result'`

Keep the highest-confidence boxes.

- Parameters: count is the maximum number of boxes to keep.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.top_boxes(frame, 2).labels)

**Practical variation**

In [ ]:
print(post.top_boxes(frame, 1).labels)

**Expected error and recovery**

In [ ]:
try:
    post.top_boxes(frame, -1)
except ValueError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.threshold

`postprocessing.threshold(minimum: 'float') -> 'Operation'`

Build a reusable score-filter operation.

- Parameters: minimum is the inclusive score threshold.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.threshold(0.5)(frame).labels)

**Practical variation**

In [ ]:
print(post.threshold(0.8)(frame).labels)

**Expected error and recovery**

In [ ]:
try:
    post.threshold('bad')(frame)
except ValueError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.clip

`postprocessing.clip(image_size: 'Optional[Tuple[int, int]]' = None) -> 'Operation'`

Build a reusable box-clipping operation.

- Parameters: image_size optionally overrides source dimensions.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.clip()(frame)[0].box)

**Practical variation**

In [ ]:
print(post.clip((80, 60))(frame)[0].box)

**Expected error and recovery**

In [ ]:
try:
    post.clip()(object())
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.remove_invalid

`postprocessing.remove_invalid(min_area: 'float' = 0) -> 'Operation'`

Build a reusable invalid-box removal operation.

- Parameters: min_area is the smallest retained box area.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(len(post.remove_invalid(min_area=100)(frame)))

**Practical variation**

In [ ]:
print(len(post.remove_invalid(2000)(frame)))

**Expected error and recovery**

In [ ]:
try:
    post.remove_invalid('bad')(frame)
except ValueError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.nms

`postprocessing.nms(iou: 'float' = 0.5, class_agnostic: 'bool' = False) -> 'Operation'`

Build a reusable nonmaximum-suppression operation.

- Parameters: iou is overlap threshold; class_agnostic optionally ignores class IDs.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.nms(iou=0.5)(frame).labels)

**Practical variation**

In [ ]:
print(post.nms(0.5, class_agnostic=True)(frame).labels)

**Expected error and recovery**

In [ ]:
try:
    post.nms(iou=1.2)(frame)
except ValueError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.top

`postprocessing.top(count: 'int') -> 'Operation'`

Build a reusable top-score operation.

- Parameters: count is the maximum number of boxes to keep.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.top(2)(frame).labels)

**Practical variation**

In [ ]:
print(post.top(1)(frame).labels)

**Expected error and recovery**

In [ ]:
try:
    post.top(-1)(frame)
except ValueError as error:
    print(type(error).__name__, 'Correct the input and retry')

## Put it together

In [ ]:
print(post.top_boxes(post.suppress(post.filter_score(frame, 0.5), iou=0.5), 1).labels)

## Remember

Postprocessing returns new results and keeps frame streams lazy. Continue with [Core](Document_Postprocessing_Core.ipynb) or the [overview](Document_Postprocessing_Overview.ipynb). Related guides: [Outputs](../outputs/Document_Outputs_Overview.ipynb) and [Visual](../visual/Document_Visual_Overview.ipynb).

In [ ]:
tutorial_workspace.cleanup()